### Instrucciones del proyecto
1. Descarga y prepara los datos. Explica el procedimiento.
2. Entrena y prueba el modelo para cada región en geo_data_0.csv:
2.1. Divide los datos en un conjunto de entrenamiento y un conjunto de validación en una proporción de 75:25
2.2. Entrena el modelo y haz predicciones para el conjunto de validación.
2.3. Guarda las predicciones y las respuestas correctas para el conjunto de validación.
2.4. Muestra el volumen medio de reservas predicho y RMSE del modelo.
2.5. Analiza los resultados.
2.6. Coloca todos los pasos previos en funciones, realiza y ejecuta los pasos 2.1-2.5 para los archivos 'geo_data_1.csv' y 'geo_data_2.csv'.
3. Prepárate para el cálculo de ganancias:
3.1. Almacena todos los valores necesarios para los cálculos en variables separadas.
3.2. Dada la inversión de 100 millones por 200 pozos petrolíferos, de media un pozo petrolífero debe producir al menos un valor de 500,000 dólares en unidades para evitar pérdidas (esto es equivalente a 111.1 unidades). Compara esta cantidad con la cantidad media de reservas en cada región.
3.3. Presenta conclusiones sobre cómo preparar el paso para calcular el beneficio.
4. Escribe una función para calcular la ganancia de un conjunto de pozos de petróleo seleccionados y modela las predicciones:
4.1. Elige los 200 pozos con los valores de predicción más altos de cada una de las 3 regiones (es decir, archivos 'csv').
4.2. Resume el volumen objetivo de reservas según dichas predicciones. Almacena las predicciones para los 200 pozos para cada una de las 3 regiones.
4.3. Calcula la ganancia potencial de los 200 pozos principales por región. Presenta tus conclusiones: propón una región para el desarrollo de pozos petrolíferos y justifica tu elección.
5. Calcula riesgos y ganancias para cada región:
5.1. Utilizando las predicciones que almacenaste en el paso 4.2, emplea la técnica del bootstrapping con 1000 muestras para hallar la distribución de los beneficios.
5.2. Encuentra el beneficio promedio, el intervalo de confianza del 95% y el riesgo de pérdidas. La pérdida es una ganancia negativa, calcúlala como una probabilidad y luego exprésala como un porcentaje.
5.3. Presenta tus conclusiones: propón una región para el desarrollo de pozos petrolíferos y justifica tu elección. ¿Coincide tu elección con la elección anterior en el punto 4.3?

Entender el problema:

La empresa tiene 100 millones de dólares para perforar 200 pozos. 
Tengo 3 datasets, uno para cada región, con información de MUCHOS pozos.
El objetivo es elegir la región que genere el beneficio promedio más alto con un riesgo de pérdidas inferior al 2.5%.

Solución propuesta:
Entrenar un modelo de regresión lineal para cada región dividiendo la muestra en 75/25 (entrenamiento/validación).
Verificar la calidad del modelo con el error RMS o R^2

Utilizar la técnica de bootstrap para estimar la distribución del beneficio promedio de cada región. Luego, comparar las distribuciones para elegir la región con el mayor beneficio promedio y un riesgo de pérdidas inferior al 2.5%.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
def load_data(ruta_a_la_carpeta: str, nombre_del_archivo_csv: str) -> pd.DataFrame:
    ruta_completa = os.path.join(ruta_a_la_carpeta, nombre_del_archivo_csv)
    data = pd.read_csv(ruta_completa)
    return data

In [ ]:
def split_data(data: pd.DataFrame, features_cols: list[str], target_col: str) -> dict[str, pd.DataFrame]:
    features = data[features_cols]
    target = data[target_col] # target es el volumen de reservas (miles de barriles)
    x_train, x_valid, y_train, y_valid = train_test_split(features, target, test_size=0.25, random_state = 12345)
    return {"x_train": x_train, "x_valid": x_valid, "y_train": y_train, "y_valid": y_valid}

In [ ]:
def modelar(data: dict[str, pd.DataFrame]) -> dict:
    
    model = LinearRegression()
    model.fit(data["x_train"], data["y_train"])
    y_predicted = model.predict(data["x_valid"])
    score = model.score(data["x_valid"], data["y_valid"])
    mse = mean_squared_error(data["y_valid"], y_predicted)
    r2 = r2_score(data["y_valid"], y_predicted)
    y_predicted_mean = np.mean(y_predicted)

    return {
        "x_valid": data["x_valid"],
        "y_valid": data["y_valid"],
        "y_predicted": y_predicted,
        "y_predicted_mean": y_predicted_mean,
        "score": score,
        "mse": mse,
        "r2": r2
    }